<a href="https://colab.research.google.com/github/kausarfatima2626/recylink1/blob/main/RecyLink_Material_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json

# 1. Define Visual Scrap Material Classes (Folder names)
MATERIAL_CLASSES = [
    "pcb",
    "cable",
    "battery",
    "crt",
    "lcd_display",
    "motor",
    "magnet_assembly",
    "mixed_plastics",
    "other"
]

# 2. Map Visual Materials to Regulatory E-Waste Categories
MATERIAL_TO_CATEGORY_MAP = {
    "pcb": "Information Technology and Telecommunication Equipment (ITE)",
    "cable": "Information Technology and Telecommunication Equipment (ITE)",
    "battery": "Small Electrical and Electronic Equipment",
    "crt": "Consumer Electrical and Electronics and Photovoltaic Panels",
    "lcd_display": "Consumer Electrical and Electronics and Photovoltaic Panels",
    "motor": "Large Electrical and Electronic Equipment",
    "magnet_assembly": "Electrical and Electronic Tools",
    "mixed_plastics": "Small Electrical and Electronic Equipment",
    "other": "Toys, Leisure, and Sports Equipment"
}

# 3. Create Colab Directory Structure
BASE_DIR = "/content/recylink_dataset"
SPLITS = ["train", "val", "test"]

for split in SPLITS:
    for cls in MATERIAL_CLASSES:
        os.makedirs(os.path.join(BASE_DIR, split, cls), exist_ok=True)

print("✅ Directory Structure Created Successfully!")
print(f"Total Visual Classes: {len(MATERIAL_CLASSES)}")
print("\nSample Mapping check:")
for mat, cat in list(MATERIAL_TO_CATEGORY_MAP.items())[:3]:
    print(f"  • {mat.upper()} ---> {cat}")

✅ Directory Structure Created Successfully!
Total Visual Classes: 9

Sample Mapping check:
  • PCB ---> Information Technology and Telecommunication Equipment (ITE)
  • CABLE ---> Information Technology and Telecommunication Equipment (ITE)
  • BATTERY ---> Small Electrical and Electronic Equipment


In [ ]:
import cv2
import numpy as np
import os
import random

BASE_DIR = "/content/recylink_dataset"
MATERIAL_CLASSES = ["pcb", "cable", "battery", "crt", "lcd_display", "motor", "magnet_assembly", "mixed_plastics", "other"]

# Create 15 sample dummy images per class inside train folder for testing pipeline
print("📦 Generating sample dataset files for testing...")
for cls in MATERIAL_CLASSES:
    cls_path = os.path.join(BASE_DIR, "train", cls)
    os.makedirs(cls_path, exist_ok=True)

    for i in range(15):
        # Generate random image (224x224x3)
        img = np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8)
        img_path = os.path.join(cls_path, f"{cls}_sample_{i}.jpg")
        cv2.imwrite(img_path, img)

# Intentionally create 1 corrupted image to test cleaning script
corrupt_path = os.path.join(BASE_DIR, "train", "pcb", "corrupt_image.jpg")
with open(corrupt_path, "w") as f:
    f.write("This is a broken file, not an image!")

print("✅ Sample files generated (including 1 intentionally broken image for quality testing).")

📦 Generating sample dataset files for testing...
✅ Sample files generated (including 1 intentionally broken image for quality testing).


In [ ]:
import os
from PIL import Image

def inspect_and_clean_dataset(base_dir):
    print("🔍 Starting Dataset Inspection & Cleaning...\n")
    report = {}
    corrupted_count = 0

    for split in ["train", "val", "test"]:
        report[split] = {}
        split_path = os.path.join(base_dir, split)

        if not os.path.exists(split_path):
            continue

        for cls in os.listdir(split_path):
            cls_folder = os.path.join(split_path, cls)
            if not os.path.isdir(cls_folder):
                continue

            valid_images = 0
            for img_name in os.listdir(cls_folder):
                img_path = os.path.join(cls_folder, img_name)

                # Check 1: File integrity check
                try:
                    with Image.open(img_path) as img:
                        img.verify()  # Verify image integrity
                    # Re-open to confirm RGB conversion readable
                    with Image.open(img_path) as img:
                        _ = img.convert("RGB")
                    valid_images += 1
                except Exception as e:
                    print(f"⚠️ Removing corrupt image: {img_path} | Reason: {e}")
                    os.remove(img_path)
                    corrupted_count += 1

            report[split][cls] = valid_images

    print(f"\n🧹 Cleaning Complete! Removed {corrupted_count} corrupted/unusable file(s).")
    return report

# Run inspection
data_report = inspect_and_clean_dataset("/content/recylink_dataset")

print("\n📊 Class Distribution Report:")
for split, counts in data_report.items():
    print(f"\n--- {split.upper()} SET ---")
    for cls_name, count in counts.items():
        print(f"  • {cls_name:15s}: {count} valid images")

🔍 Starting Dataset Inspection & Cleaning...

⚠️ Removing corrupt image: /content/recylink_dataset/train/pcb/corrupt_image.jpg | Reason: cannot identify image file '/content/recylink_dataset/train/pcb/corrupt_image.jpg'

🧹 Cleaning Complete! Removed 1 corrupted/unusable file(s).

📊 Class Distribution Report:

--- TRAIN SET ---
  • pcb            : 15 valid images
  • lcd_display    : 15 valid images
  • other          : 15 valid images
  • battery        : 15 valid images
  • motor          : 15 valid images
  • magnet_assembly: 15 valid images
  • mixed_plastics : 15 valid images
  • cable          : 15 valid images
  • crt            : 15 valid images

--- VAL SET ---
  • pcb            : 0 valid images
  • lcd_display    : 0 valid images
  • other          : 0 valid images
  • battery        : 0 valid images
  • motor          : 0 valid images
  • magnet_assembly: 0 valid images
  • mixed_plastics : 0 valid images
  • cable          : 0 valid images
  • crt            : 0 valid image

In [ ]:
import os
import shutil
import random

def create_stratified_splits(base_dir, train_ratio=0.70, val_ratio=0.15):
    print("🔀 Splitting dataset into Train / Val / Test (70% - 15% - 15%)...")

    # We collect all current images from train folder
    train_base = os.path.join(base_dir, "train")
    classes = [c for c in os.listdir(train_base) if os.path.isdir(os.path.join(train_base, c))]

    for cls in classes:
        cls_dir = os.path.join(train_base, cls)
        images = os.listdir(cls_dir)
        random.shuffle(images)

        total = len(images)
        train_end = int(total * train_ratio)
        val_end = train_end + int(total * val_ratio)

        val_imgs = images[train_end:val_end]
        test_imgs = images[val_end:]

        # Move to Val folder
        val_target_dir = os.path.join(base_dir, "val", cls)
        os.makedirs(val_target_dir, exist_ok=True)
        for img in val_imgs:
            shutil.move(os.path.join(cls_dir, img), os.path.join(val_target_dir, img))

        # Move to Test folder
        test_target_dir = os.path.join(base_dir, "test", cls)
        os.makedirs(test_target_dir, exist_ok=True)
        for img in test_imgs:
            shutil.move(os.path.join(cls_dir, img), os.path.join(test_target_dir, img))

    print("✅ Split process complete!")

# Run splitting logic
create_stratified_splits("/content/recylink_dataset")

# Verify final folder counts
for split in ["train", "val", "test"]:
    total_imgs = sum([len(os.listdir(os.path.join("/content/recylink_dataset", split, c)))
                      for c in MATERIAL_CLASSES if os.path.exists(os.path.join("/content/recylink_dataset", split, c))])
    print(f"Total images in {split.upper()} set: {total_imgs}")

🔀 Splitting dataset into Train / Val / Test (70% - 15% - 15%)...
✅ Split process complete!
Total images in TRAIN set: 90
Total images in VAL set: 18
Total images in TEST set: 27


In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader
import os

# Device setup (GPU check)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Using device: {device}")

BASE_DIR = "/content/recylink_dataset"
BATCH_SIZE = 16
NUM_CLASSES = len(MATERIAL_CLASSES)

# 1. Preprocessing & Data Augmentation Pipeline (Task 4)
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]),
}

# 2. PyTorch Datasets & DataLoaders Creation
image_datasets = {
    x: datasets.ImageFolder(os.path.join(BASE_DIR, x), data_transforms[x])
    for x in ['train', 'val', 'test']
}

dataloaders = {
    x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE, shuffle=(x == 'train'), num_workers=2)
    for x in ['train', 'val', 'test']
}

print("✅ Preprocessing & DataLoaders Pipeline Ready!")
print(f"Class Mapping: {image_datasets['train'].class_to_idx}")

# 3. MobileNetV2 Transfer Learning Baseline Architecture (Task 5)
def build_mobilenet_v2(num_classes):
    # Pretrained MobileNetV2 weights load karo
    weights = models.MobileNet_V2_Weights.DEFAULT
    model = models.mobilenet_v2(weights=weights)

    # Pretrained backbone features freeze kar do
    for param in model.parameters():
        param.requires_grad = False

    # Classifier head rewrite karo customized e-waste material classes ke liye
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2),
        nn.Linear(in_features, 128),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(128, num_classes)
    )
    return model

model = build_mobilenet_v2(NUM_CLASSES).to(device)
print("\n✅ MobileNetV2 Baseline Model Successfully Built & Loaded to GPU/CPU!")

⚡ Using device: cuda
✅ Preprocessing & DataLoaders Pipeline Ready!
Class Mapping: {'battery': 0, 'cable': 1, 'crt': 2, 'lcd_display': 3, 'magnet_assembly': 4, 'mixed_plastics': 5, 'motor': 6, 'other': 7, 'pcb': 8}
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 131MB/s]



✅ MobileNetV2 Baseline Model Successfully Built & Loaded to GPU/CPU!


In [ ]:
import torch.optim as optim
import time
import copy

# Loss function & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

def train_model(model, criterion, optimizer, num_epochs=5):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    print("🚀 Starting Training Loop...\n")

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        print("-" * 25)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(image_datasets[phase])
            epoch_acc = running_corrects.double() / len(image_datasets[phase])

            print(f"{phase.upper()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

            # Save best weights
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f"⏱️ Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
    print(f"🏆 Best Validation Accuracy: {best_acc:4f}")

    # Load best weights
    model.load_state_dict(best_model_wts)
    return model

# Model train karo (Default: 5 epochs)
trained_model = train_model(model, criterion, optimizer, num_epochs=5)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

def evaluate_model(model, dataloader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    class_names = [MATERIAL_CLASSES[i] for i in sorted(image_datasets['test'].class_to_idx.values())]

    # 1. Print Precision, Recall, F1-Score
    print("📊 CLASSIFICATION REPORT:")
    print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

    # 2. Plot Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted Material')
    plt.ylabel('Actual Material')
    plt.title('E-Waste Material Confusion Matrix')
    plt.show()

# Run Evaluation on Test Set
evaluate_model(trained_model, dataloaders['test'])

In [ ]:
import torch
import json
import os

# Create export directory
EXPORT_DIR = "/content/exported_model"
os.makedirs(EXPORT_DIR, exist_ok=True)

MODEL_PATH = os.path.join(EXPORT_DIR, "mobilenet_ewaste_classifier.pth")
CONFIG_PATH = os.path.join(EXPORT_DIR, "class_config.json")

# 1. Save PyTorch Model Weights
torch.save(trained_model.state_dict(), MODEL_PATH)
print(f"✅ Model weights saved at: {MODEL_PATH}")

# 2. Save Class Index & Regulatory Mapping JSON
class_to_idx = image_datasets['train'].class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}

export_config = {
    "idx_to_class": idx_to_class,
    "material_to_category_map": MATERIAL_TO_CATEGORY_MAP
}

with open(CONFIG_PATH, "w") as f:
    json.dump(export_config, f, indent=4)

print(f"✅ Class configuration and mappings saved at: {CONFIG_PATH}")

In [15]:
import zipfile
import os

zip_path = "/archivewaste.zip"
extract_path = "/content/real_dataset"

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("✅ `archivewaste.zip` successfully unzipped!")
    print("Extracted items:", os.listdir(extract_path))
else:
    print("⚠️ `archivewaste.zip` /content/ folder mein nahi mila. Direct files section mein zip upload karein.")

✅ `archivewaste.zip` successfully unzipped!
Extracted items: ['modified-dataset']


In [16]:
import os

# Auto-detect source directory path
extracted_items = os.listdir(extract_path)
if len(extracted_items) == 1 and os.path.isdir(os.path.join(extract_path, extracted_items[0])):
    SOURCE_DIR = os.path.join(extract_path, extracted_items[0])
else:
    SOURCE_DIR = extract_path

BASE_DIR = "/content/recylink_dataset"

# Person 1 Target Material Classes
MATERIAL_CLASSES = ["pcb", "cable", "battery", "crt", "lcd_display", "motor", "magnet_assembly", "mixed_plastics", "other"]

# Folder Mapping Dictionary
FOLDER_MAP = {
    "PCB": "pcb",
    "pcb": "pcb",
    "Battery": "battery",
    "battery": "battery",
    "Keyboard": "cable",
    "keyboard": "cable",
    "Mouse": "cable",
    "mouse": "cable",
    "Mobile": "lcd_display",
    "mobile": "lcd_display",
    "Television": "crt",
    "television": "crt",
    "Microwave": "motor",
    "microwave": "motor",
    "Washing Machine": "motor",
    "washing_machine": "motor",
    "Player": "magnet_assembly",
    "player": "magnet_assembly",
    "Printer": "mixed_plastics",
    "printer": "mixed_plastics",
    "plastic": "mixed_plastics",
    "cardboard": "other",
    "glass": "other",
    "metal": "other",
    "organic": "other",
    "paper": "other",
    "trash": "other"
}

# Create Train, Val, Test directories
for split in ["train", "val", "test"]:
    for cls in MATERIAL_CLASSES:
        os.makedirs(os.path.join(BASE_DIR, split, cls), exist_ok=True)

print("✅ Target directory structure ready!")
print("Source Folder Path:", SOURCE_DIR)

✅ Target directory structure ready!
Source Folder Path: /content/real_dataset/modified-dataset


In [17]:
import shutil

copied_count = 0
found_folders = os.listdir(SOURCE_DIR)

for folder_name in found_folders:
    src_folder_path = os.path.join(SOURCE_DIR, folder_name)
    if os.path.isdir(src_folder_path):
        target_class = FOLDER_MAP.get(folder_name, FOLDER_MAP.get(folder_name.lower(), "other"))
        target_train_path = os.path.join(BASE_DIR, "train", target_class)

        for img_name in os.listdir(src_folder_path):
            src_img = os.path.join(src_folder_path, img_name)
            if os.path.isfile(src_img):
                dst_img = os.path.join(target_train_path, f"{folder_name}_{img_name}")
                shutil.copy(src_img, dst_img)
                copied_count += 1

print(f"✅ {copied_count} real images organized for training!")

✅ 0 real images organized for training!


In [18]:
from PIL import Image
import random

# 1. Clean Corrupt Images
corrupted_count = 0
for cls in MATERIAL_CLASSES:
    cls_folder = os.path.join(BASE_DIR, "train", cls)
    for img_name in os.listdir(cls_folder):
        img_path = os.path.join(cls_folder, img_name)
        try:
            with Image.open(img_path) as img:
                img.verify()
        except Exception:
            os.remove(img_path)
            corrupted_count += 1

print(f"🧹 Removed {corrupted_count} corrupted images.")

# 2. Perform Split (Train: 70%, Val: 15%, Test: 15%)
for cls in MATERIAL_CLASSES:
    cls_dir = os.path.join(BASE_DIR, "train", cls)
    images = os.listdir(cls_dir)
    random.shuffle(images)

    total = len(images)
    train_end = int(total * 0.70)
    val_end = train_end + int(total * 0.15)

    val_imgs = images[train_end:val_end]
    test_imgs = images[val_end:]

    for img in val_imgs:
        shutil.move(os.path.join(cls_dir, img), os.path.join(BASE_DIR, "val", cls, img))

    for img in test_imgs:
        shutil.move(os.path.join(cls_dir, img), os.path.join(BASE_DIR, "test", cls, img))

print("\n📊 Dataset Split Complete:")
for split in ["train", "val", "test"]:
    total_imgs = sum([len(os.listdir(os.path.join(BASE_DIR, split, c))) for c in MATERIAL_CLASSES])
    print(f"  • {split.upper()} Set: {total_imgs} images")

🧹 Removed 0 corrupted images.

📊 Dataset Split Complete:
  • TRAIN Set: 0 images
  • VAL Set: 0 images
  • TEST Set: 0 images


In [19]:
import os
import shutil

# 1. Unzipped Path Inspector
extract_path = "/content/real_dataset"

print("🔍 Checking extracted directory contents...")
all_found_folders = []

for root, dirs, files in os.walk(extract_path):
    for d in dirs:
        full_dir = os.path.join(root, d)
        # Check if directory contains files directly
        contained_files = [f for f in os.listdir(full_dir) if os.path.isfile(os.path.join(full_dir, f))]
        if len(contained_files) > 0:
            all_found_folders.append((d, full_dir, len(contained_files)))

print(f"\n✅ Total subfolders with images found: {len(all_found_folders)}")
for folder_name, full_path, file_count in all_found_folders:
    print(f"  • Folder: '{folder_name}' | Files: {file_count} | Path: {full_path}")

# 2. Re-populate Images automatically to target train directory
BASE_DIR = "/content/recylink_dataset"
MATERIAL_CLASSES = ["pcb", "cable", "battery", "crt", "lcd_display", "motor", "magnet_assembly", "mixed_plastics", "other"]

# Standardized folder mapping
FOLDER_MAP = {
    "pcb": "pcb", "battery": "battery", "keyboard": "cable", "mouse": "cable",
    "mobile": "lcd_display", "television": "crt", "microwave": "motor",
    "washing machine": "motor", "player": "magnet_assembly", "printer": "mixed_plastics",
    "plastic": "mixed_plastics", "cardboard": "other", "glass": "other",
    "metal": "other", "organic": "other", "paper": "other", "trash": "other"
}

copied_images = 0
for folder_name, full_path, file_count in all_found_folders:
    # Match lowercase folder name
    clean_name = folder_name.lower().strip()
    target_cls = FOLDER_MAP.get(clean_name, "other")

    target_train_dir = os.path.join(BASE_DIR, "train", target_cls)
    os.makedirs(target_train_dir, exist_ok=True)

    for file in os.listdir(full_path):
        src_file_path = os.path.join(full_path, file)
        if os.path.isfile(src_file_path) and file.lower().endswith(('.jpg', '.jpeg', '.png')):
            dst_file_path = os.path.join(target_train_dir, f"{clean_name}_{file}")
            shutil.copy(src_file_path, dst_file_path)
            copied_images += 1

print(f"\n🎉 Total {copied_images} images successfully copied to train directory!")

🔍 Checking extracted directory contents...

✅ Total subfolders with images found: 30
  • Folder: 'Mouse' | Files: 30 | Path: /content/real_dataset/modified-dataset/test/Mouse
  • Folder: 'Mobile' | Files: 30 | Path: /content/real_dataset/modified-dataset/test/Mobile
  • Folder: 'PCB' | Files: 30 | Path: /content/real_dataset/modified-dataset/test/PCB
  • Folder: 'Keyboard' | Files: 30 | Path: /content/real_dataset/modified-dataset/test/Keyboard
  • Folder: 'Player' | Files: 30 | Path: /content/real_dataset/modified-dataset/test/Player
  • Folder: 'Television' | Files: 30 | Path: /content/real_dataset/modified-dataset/test/Television
  • Folder: 'Washing Machine' | Files: 30 | Path: /content/real_dataset/modified-dataset/test/Washing Machine
  • Folder: 'Microwave' | Files: 30 | Path: /content/real_dataset/modified-dataset/test/Microwave
  • Folder: 'Printer' | Files: 30 | Path: /content/real_dataset/modified-dataset/test/Printer
  • Folder: 'Battery' | Files: 30 | Path: /content/real_d